# Motivación: Algoritmos de Búsqueda Informada

En las sesiones anteriores estudiamos los **algoritmos de búsqueda no informada** (o *ciegos*), como **BFS (Breadth-First Search)**, **DFS (Depth-First Search)** y **UCS (Uniform-Cost Search)**.  

Estos algoritmos trabajan únicamente con la **información explícita del problema**:
- **Estados:** los nodos del espacio de búsqueda.
- **Acciones:** transiciones posibles entre estados.
- **Costos (opcional):** si existen, como en UCS.

En otras palabras, el algoritmo solo conoce la **estructura del grafo** que modela el problema:  
$$
\text{estados, acciones, costos (si existen), estado inicial y estados objetivo}.
$$

Bajo este esquema:
- **BFS** explora el grafo **por niveles**, garantizando la solución con **menor número de pasos** si todos los costos son iguales.
- **DFS** explora un **camino completo** hasta el final antes de retroceder, usando menos memoria pero sin garantizar la solución más corta.
- **UCS** utiliza los **costos reales** de las acciones para garantizar la **solución de costo mínimo**, pero sigue sin información sobre la **dirección** hacia la meta.

---

El problema principal:  
- Estos algoritmos pueden **explorar muchas regiones irrelevantes** del grafo antes de llegar al objetivo.  
- El **tiempo y la memoria** pueden crecer rápidamente si el espacio de estados es muy grande.

---

La **búsqueda informada** intenta resolver esto usando una **función heurística** \( h(n) \) que estima la distancia o costo desde el nodo \( n \) hasta la meta, permitiendo **priorizar** los nodos que parecen **más prometedores** y reducir la exploración innecesaria.


## Hacia los Algoritmos de Búsqueda Informada

Los **algoritmos de búsqueda informada** amplían la información usada en la búsqueda: además de la estructura del grafo (estados, acciones, costos), incorporan una **función heurística** \( h(n) \) que estima la distancia o costo desde el estado actual \( n \) hasta el objetivo.  

Con esta información adicional, la búsqueda puede **priorizar** qué nodos explorar, dirigiéndose hacia regiones que **parecen más cercanas** a la meta y reduciendo el tiempo y la memoria necesarios.

En este Notebook estudiaremos dos algoritmos clásicos:  

1. **Búsqueda Voraz (Greedy Best-First Search):**  
   - Expande siempre el nodo que **parece más cercano** al objetivo según \( h(n) \).  
   - Es rápida, pero no garantiza encontrar la solución óptima.

2. **Búsqueda A\* (A-Star):**  
   - Combina el **costo real acumulado** \( g(n) \) con la **estimación heurística** \( h(n) \) usando  
     \[
     f(n) = g(n) + h(n)
     \]
   - Encuentra soluciones **óptimas** si \( h(n) \) es **admisible** (no sobreestima el costo real).

Con estos algoritmos podemos:  
- Reducir la **cantidad de nodos explorados**,  
- Encontrar soluciones **más rápido**,  
- Mantener, en algunos casos, la **optimalidad** que tenían UCS o BFS, pero con mucho mejor eficiencia.

**Ejemplo:** Considere el mapa de la imagen. Supón que queremos encontrar la mejor ruta entre Lanoi y Kosos. Usaremos este mapa para ilustrar la implementación y funcionamiento de los algorítmos de búsqueda no informada. 

<img src="Ciudades.png" alt="Grafo" width="750"/>

In [27]:
#debemos represenar el conjuntos de acciones posibles en la exploración y el conjunto de estados posibles 
class Accion:
    def __init__(self, nombre):
        self.nombre = nombre
    def __str__(self):
        return self.nombre

#definimos la clase de estados 
class Estado:
    def __init__(self, estado, acciones):
        self.nombre = estado 
        self.acciones = acciones 
    def __str__(self):
        return self.nombre
accN = Accion('N')
accS = Accion('S')
accE = Accion('E')
accO = Accion('O')
accNE = Accion('NE')
accNO = Accion('NO')
accSE = Accion('SE')
accSO = Accion('SO')
#crear los estados 
lanoi = Estado('Lanoi', [accNE])
nahoi = Estado('Nahoi', [accSO, accNO, accNE])
ruun = Estado('Ruun', [accNO, accNE, accE, accSE])
milos = Estado('Milos', [accO, accSO, accN])
ghildo = Estado('Ghildo', [accN, accE, accSE])
kuart = Estado('Kuart', [accO, accSO, accNE])
boomon = Estado('Boomon', [accN, accSO])
goorum = Estado('Goorum', [accO, accS])
shiphos = Estado('Shiphos', [accO, accE])
nokshos = Estado('Nokshos', [accNO, accS, accE])
pharis = Estado('Pharis', [accNO, accSO])
khamin = Estado('Khamin', [accSE, accNO, accO])
tarios = Estado('Tarios', [accO, accNO, accNE, accE])
peranna = Estado('Peranna', [accO, accE])
khandon = Estado('Khandon', [accO, accS])
tawa = Estado('Tawa', [accSO, accSE, accNE])
theer = Estado('Theer', [accSO, accSE])
roria = Estado('Roria', [accNO, accSO, accE])
kosos = Estado('Kosos', [accO])
acciones = {'Lanoi':{'NE':nahoi},
            'Nahoi':{'SO':lanoi, 'NO':ruun, 'NE':milos},
            'Ruun':{'NO':ghildo, 'NE':kuart, 'E':milos, 'SE':nahoi },
            'Milos':{'O':ruun, 'SO':nahoi, 'N':khandon},
            'Ghildo':{'N':nokshos, 'E':kuart, 'SE':ruun},
            'Kuart':{'O':ghildo, 'SO':ruun, 'NE':boomon},
            'Boomon':{'N':goorum, 'SO':kuart},
            'Goorum':{'O':shiphos, 'S':boomon},
            'Shiphos':{'O':nokshos, 'E':goorum},
            'Nokshos':{'NO':pharis, 'S':ghildo, 'E':shiphos},
            'Pharis':{'NO':khamin, 'SO':nokshos},
            'Khamin':{'SE':pharis, 'NO':tawa, 'NE':tarios},
            'Tarios':{'O':khamin, 'NO':tawa, 'NE':roria, 'E':peranna},
            'Peranna':{'O':tarios, 'E':khandon},
            'Khandon':{'O':peranna, 'S':milos},
            'Tawa':{'SO':khamin, 'SE':tarios, 'NE':theer},
            'Theer':{'SO':tawa, 'SE':roria},
            'Roria':{'NO':theer, 'SO':tarios, 'E':kosos},
            'Kosos':{'O':roria}}
costos = {'Lanoi':{'NE':42},
        'Nahoi':{'SO':42, 'NO':21, 'NE':95},
        'Ruun':{'NO':88, 'NE':16, 'E':90, 'SE':21 },
        'Milos':{'O':90, 'SO':95, 'N':133},
        'Ghildo':{'N':17, 'E':92, 'SE':88},
        'Kuart':{'O':92, 'SO':16, 'NE':83},
        'Boomon':{'N':8, 'SO':83},
        'Goorum':{'O':59, 'S':8},
        'Shiphos':{'O':71, 'E':59},
        'Nokshos':{'NO':5, 'S':17, 'E':71},
        'Pharis':{'NO':29, 'SO':5},
        'Khamin':{'SE':29, 'NO':121, 'NE':98},
        'Tarios':{'O':98, 'NO':83, 'NE':57, 'E':82},
        'Peranna':{'O':82, 'E':44},
        'Khandon':{'O':44, 'S':133},
        'Tawa':{'SO':121, 'SE':83, 'NE':11},
        'Theer':{'SO':11, 'SE':36},
        'Roria':{'NO':36, 'SO':57, 'E':104},
        'Kosos':{'O':104}}

## Búsqueda Voraz (Greedy Best-First Search)

La **búsqueda voraz** es un algoritmo de búsqueda informada que utiliza **únicamente** la función heurística \( h(n) \) para guiar la exploración:

$$
f(n) = h(n)
$$

En cada paso, el algoritmo expande el nodo que **parece más cercano al objetivo** según la estimación \( h(n) \).  
De esta forma, la búsqueda se orienta directamente hacia la meta, intentando minimizar el número de nodos explorados.

---

#### Características principales

- **Estrategia:** Expande el nodo con el valor \( h(n) \) más bajo en la frontera.
- **Estructura de datos:** Cola de prioridad (min-heap) ordenada por \( h(n) \).
- **Complejidad:** Depende de la calidad de \( h(n) \); en el peor caso puede ser exponencial.
- **Óptima:** No, porque ignora el costo real \( g(n) \); puede encontrar soluciones que no son de costo mínimo.
- **Completa:** No siempre; puede quedar atrapada en bucles si no se maneja correctamente.

---

#### Ventajas
- Suele ser **más rápida** que la búsqueda no informada, porque explora menos nodos.
- Si la heurística es buena, puede llegar a la solución rápidamente.

#### Desventajas
- **No garantiza** la solución óptima.
- Puede desviarse si la heurística no es confiable.

---

#### Intuición
La búsqueda voraz es como seguir una **luz en la distancia**: siempre avanzamos hacia donde **parece** estar la meta, sin preocuparnos por los costos ya recorridos ni por verificar si realmente es el camino más corto.


#### Seudocódigo de Búsqueda Voraz (Greedy Best-First Search)

```
Greedy(problema):
    raiz ← Nodo(problema.estado_inicial)
    raiz.heuristica ← h(raiz.estado)
    si problema.es_objetivo(raiz.estado):
        retornar raiz

    frontera ← cola_prioridad_min()        # prioridad = h(n)
    insertar(frontera, (h(raiz.estado), raiz))
    visitados ← { }                        # opcional: controlar duplicados
    best_h ← { raiz.estado : h(raiz.estado) }  # opcional

    mientras frontera no esté vacía:
        (_, n) ← extraer_min(frontera)     # nodo con menor h(n)

        si problema.es_objetivo(n.estado):
            retornar n

        marcar n.estado como visitado      # opcional

        para cada (accion, s) en problema.sucesores(n.estado):
            si s no está en visitados:
                h_s ← h(s)                 # heurística del sucesor
                hijo ← Nodo(estado = s, padre = n, accion = accion, heuristica = h_s)
                insertar(frontera, (h_s, hijo))

    retornar fallo   # no existe solución
```


In [28]:
#redefinimos Nodo y Problema para incluir heurísticas 
#definir acá, la primera versión, de la clase Nodo
class Nodo:
    """ La clade nodo tendrá como atributos
    estado: estado en el ábol de búsqueda
    padre: nodo padre
    accion: acción que produjo el estado actual desde el padre 
    depth: profundidad del Nodo en el árbol de busqueda"""
    def __init__(self, estado, padre=None, accion=None, costo=0.0, heuristica=0.0):
        self.estado = estado
        self.padre = padre 
        self.accion = accion 
        self.costo = costo
        self.heuristica = heuristica
        self.depth = 0 if padre is None  else  padre.depth + 1
    def reconstruir_camino(self):
        camino = []
        n = self
        while n is not None: 
            camino.append(n.estado)
            n = n.padre
        camino.reverse()
        return camino 
    def reconstruir_acciones(self):
        acciones = []
        n = self
        while n.padre is not None:
            acciones.append(n.accion)
            n = n.padre 
        acciones.reverse()
        return acciones 
    def __str__(self):
        return f'Estado: {self.estado}'





In [29]:
#incluir heurísticas Problema y método 
class Problema:
    def __init__(self, estado_inicial, estados_objetivo, acciones, costos = None, heuristicas = None):
        self.estado_inicial = estado_inicial 
        self.estados_objetivo = set(estados_objetivo)
        self.acciones = acciones 
        self.costos = costos 
        self.heuristicas = heuristicas 
    def es_objetivo(self, estado):
        return estado in self.estados_objetivo
    def sucesores(self, estado):
        """este método retornará acción-estado_siguiente """
        d = self.acciones.get(estado.nombre,{})
        return [(Accion(a), b) for a,b in d.items() ]
    def resultado(self, estado, accion):
        """retornará el nuevo estado si tomo la acción en estado"""
        d = self.acciones.get(estado.nombre, {})
        return d.get(accion.nombre, None)
    def heuristica(self, estado):
        """el metodo retorna el vaor de la heurística desde el estado dado hasta el estado objetivo"""
        return self.heuristicas.get(estado, {}).get(next(iter(self.estados_objetivo)), 10**9)
    def costo_accion(self, estado, accion):
        return self.costos.get(estado.nombre, {}).get(accion.nombre, 1.0)
    def __str__(self):
        str = f'estado inicial={self.estado_inicial}---> estados objetivo:{next(iter(self.estados_objetivo)) }'
        return str
        

In [30]:
heuristicas = {'Lanoi': {'Lanoi': 0, 'Nahoi': 32, 'Ruun': 43, 'Milos': 90, 'Ghildo': 81, 'Kuart': 50, 'Boomon': 90, 'Goorum': 95, 'Shiphos': 84,  'Nokshos': 108,
                            'Pharis': 111, 'Khamin': 124, 'Tarios': 145, 'Peranna': 157, 'Khandon': 182, 'Tawa': 180, 'Theer': 183, 'Roria': 174, 'Kosos': 224},
                'Nahoi': {'Lanoi': 32, 'Nahoi': 0, 'Ruun': 12, 'Milos': 72, 'Ghildo': 73, 'Kuart': 18, 'Boomon': 57, 'Goorum': 63, 'Shiphos': 56, 'Nokshos': 93,
                            'Pharis': 97, 'Khamin': 107, 'Tarios': 117, 'Peranna': 125, 'Khandon': 150, 'Tawa': 153, 'Theer': 154, 'Roria': 143, 'Kosos': 191},
                'Ruun': {'Lanoi': 43, 'Nahoi': 12, 'Ruun': 0, 'Milos': 71, 'Ghildo': 75, 'Kuart': 6,'Boomon': 45,'Goorum': 50,'Shiphos': 45,'Nokshos': 87,'Pharis': 89,
                        'Khamin': 100,'Tarios': 105,'Peranna': 103,'Khandon': 140,'Tawa': 140,'Theer': 142,'Roria': 131,'Kosos': 179},
                'Milos': {'Lanoi': 90,'Nahoi': 72,'Ruun': 71,'Milos': 0,'Ghildo': 145,'Kuart': 70,'Boomon': 76,'Goorum': 81,'Shiphos': 103,'Nokshos': 156,'Pharis': 157,
                            'Khamin': 165,'Tarios': 138,'Peranna': 116,'Khandon': 122,'Tawa': 176,'Theer': 174,'Roria': 159,'Kosos': 172},
                'Ghildo': {'Lanoi': 81,        'Nahoi': 73,'Ruun': 75,'Milos': 145,'Ghildo': 0,'Kuart': 75,'Boomon': 94,'Goorum': 93,'Shiphos': 61,'Nokshos': 34,
                            'Pharis': 37, 'Khamin': 53,'Tarios': 112,'Peranna': 150,'Khandon': 187,'Tawa': 134,'Theer': 140,'Roria': 136,'Kosos': 211},
                'Kuart': {'Lanoi': 50, 'Nahoi': 18,'Ruun': 6, 'Milos': 70, 'Ghildo': 75,'Kuart': 0,'Boomon': 38,'Goorum': 43,'Shiphos': 42,'Nokshos': 87,'Pharis': 89,
                            'Khamin': 98,'Tarios': 98,'Peranna': 107,'Khandon': 134, 'Tawa': 135, 'Theer': 137, 'Roria': 124,'Kosos': 173},
                'Boomon': {'Lanoi': 90,'Nahoi': 57, 'Ruun': 45, 'Milos': 76, 'Ghildo': 94, 'Kuart': 38, 'Boomon': 0,'Goorum': 6, 'Shiphos': 36, 'Nokshos': 91, 'Pharis': 91,
                            'Khamin': 95,'Tarios': 67, 'Peranna': 68, 'Khandon': 98, 'Tawa': 105, 'Theer': 104,'Roria': 91, 'Kosos': 134},
                'Goorum': {'Lanoi': 95, 'Nahoi': 63,'Ruun': 50, 'Milos': 81, 'Ghildo': 93, 'Kuart': 43,'Boomon': 6,'Goorum': 0, 'Shiphos': 33, 'Nokshos': 88,'Pharis': 87,
                            'Khamin': 92, 'Tarios': 62, 'Peranna': 64,'Khandon': 97,    'Tawa': 98, 'Theer': 98,'Roria': 85,'Kosos': 132},
                'Shiphos': {'Lanoi': 84,'Nahoi': 56, 'Ruun': 45,  'Milos': 103, 'Ghildo': 61,'Kuart': 42, 'Boomon': 36,'Goorum': 33, 'Shiphos': 0, 'Nokshos': 55, 'Pharis': 55,
                            'Khamin': 63,'Tarios': 64,'Peranna': 92, 'Khandon': 127, 'Tawa': 97, 'Theer': 101,'Roria': 92,'Kosos': 156},
                'Nokshos': {'Lanoi': 108, 'Nahoi': 93,'Ruun': 87, 'Milos': 156, 'Ghildo': 34,'Kuart': 87,'Boomon': 91, 'Goorum': 88,  'Shiphos': 55,'Nokshos': 0,'Pharis': 3,
                            'Khamin': 18, 'Tarios': 86,'Peranna': 133, 'Khandon': 171, 'Tawa': 103,'Theer': 109, 'Roria': 109, 'Kosos': 189},
                'Pharis': {'Lanoi': 111,'Nahoi': 97, 'Ruun': 89, 'Milos': 157, 'Ghildo': 37,'Kuart': 89, 'Boomon': 91,'Goorum': 87,'Shiphos': 55,'Nokshos': 3,'Pharis': 0, 'Khamin': 14,
                            'Tarios': 83, 'Peranna': 132,'Khandon': 170,'Tawa': 100,'Theer': 107, 'Roria': 105, 'Kosos': 186},
                'Khamin': {'Lanoi': 124, 'Nahoi': 107,'Ruun': 100,  'Milos': 165,  'Ghildo': 53,  'Kuart': 98, 'Boomon': 95,   'Goorum': 92, 'Shiphos': 63, 'Nokshos': 18,
                            'Pharis': 14,'Khamin': 0, 'Tarios': 77,'Peranna': 128,  'Khandon': 168, 'Tawa': 89, 'Theer': 97, 'Roria': 97, 'Kosos': 182},
                'Tarios': {'Lanoi': 145,'Nahoi': 117, 'Ruun': 105, 'Milos': 138, 'Ghildo': 112, 'Kuart': 98,'Boomon': 67,'Goorum': 62,'Shiphos': 64, 'Nokshos': 86,'Pharis': 83,
                            'Khamin': 77, 'Tarios': 0, 'Peranna': 55,'Khandon': 94, 'Tawa': 38, 'Theer': 38, 'Roria': 27, 'Kosos': 104},
                'Peranna': {'Lanoi': 157, 'Nahoi': 125, 'Ruun': 103,'Milos': 116,'Ghildo': 150, 'Kuart': 107, 'Boomon': 68, 'Goorum': 64, 'Shiphos': 92, 'Nokshos': 133,
                            'Pharis': 132,'Khamin': 128,'Tarios': 55, 'Peranna': 0,'Khandon': 38, 'Tawa': 80,'Theer': 73, 'Roria': 59,'Kosos': 66},
                'Khandon': {'Lanoi': 182,  'Nahoi': 150,'Ruun': 140,'Milos': 122, 'Ghildo': 187,'Kuart': 134, 'Boomon': 98, 'Goorum': 97, 'Shiphos': 127, 'Nokshos': 171,
                            'Pharis': 170,'Khamin': 168, 'Tarios': 94,'Peranna': 38, 'Khandon': 0, 'Tawa': 113, 'Theer': 107, 'Roria': 92, 'Kosos': 52},
                'Tawa': {'Lanoi': 180, 'Nahoi': 153,'Ruun': 140, 'Milos': 176, 'Ghildo': 134, 'Kuart': 135, 'Boomon': 105, 'Goorum': 98, 'Shiphos': 97, 'Nokshos': 103,'Pharis': 100,
                        'Khamin': 89,  'Tarios': 38, 'Peranna': 80, 'Khandon': 113,'Tawa': 0, 'Theer': 9,  'Roria': 21,'Kosos': 105},
                'Theer': {'Lanoi': 183, 'Nahoi': 154, 'Ruun': 142, 'Milos': 174,'Ghildo': 140,'Kuart': 137,  'Boomon': 104,  'Goorum': 98,'Shiphos': 101, 'Nokshos': 109,
                            'Pharis': 107, 'Khamin': 97,'Tarios': 38,'Peranna': 73, 'Khandon': 107,'Tawa': 9, 'Theer': 0, 'Roria': 16,'Kosos': 96},
                'Roria': {'Lanoi': 174, 'Nahoi': 143, 'Ruun': 131,'Milos': 159, 'Ghildo': 136, 'Kuart': 124, 'Boomon': 91, 'Goorum': 85,  'Shiphos': 92, 'Nokshos': 109,'Pharis': 105,
                            'Khamin': 97, 'Tarios': 27, 'Peranna': 59, 'Khandon': 92, 'Tawa': 21,   'Theer': 16,  'Roria': 0,'Kosos': 87},
                'Kosos': {'Lanoi': 224, 'Nahoi': 191,  'Ruun': 179, 'Milos': 172,'Ghildo': 211, 'Kuart': 173,   'Boomon': 134,'Goorum': 132, 'Shiphos': 156, 'Nokshos': 189, 'Pharis': 186,
                            'Khamin': 182, 'Tarios': 104,'Peranna': 66, 'Khandon': 52, 'Tawa': 105, 'Theer': 96, 'Roria': 87, 'Kosos': 0}}

In [31]:
#Escribamos el algorítmo de búsqueda voraz 
import heapq
from itertools import count 

def voraz(problema):
    """La funcion Voraz que hace la busqueda teniendo en cuenta la heurística h"""
    raiz = Nodo(problema.estado_inicial)
    raiz.heuristica = float(problema.heuristica(raiz.estado))
    if problema.es_objetivo(raiz.estado):
        return raiz
    frontera = []
    contar = count()  # contador para desempatar nodos con la misma heurística
    heapq.heappush(frontera, (raiz.heuristica, next(contar), raiz))
    visitados = {raiz.estado}
    while frontera:
        _, _, nodo = heapq.heappop(frontera)
        if problema.es_objetivo(nodo.estado):
            return nodo
        for accion, s in problema.sucesores(nodo.estado):
            if s in visitados:
                continue
            hijo = Nodo(s, padre=nodo, accion=accion)
            hijo.heuristica = float(problema.heuristica(hijo.estado))
            visitados.add(hijo.estado)
            heapq.heappush(frontera, (hijo.heuristica, next(contar), hijo))
    return None
            
            


In [32]:
Pr1= Problema(lanoi,{kosos}, acciones, costos, heuristicas=heuristicas)
str(Pr1)

'estado inicial=Lanoi---> estados objetivo:Kosos'

In [33]:
Sol_voraz = voraz(Pr1)
camino_v = [str(i) for i in Sol_voraz.reconstruir_camino()]
camino_v

['Lanoi', 'Nahoi', 'Milos', 'Khandon', 'Peranna', 'Tarios', 'Roria', 'Kosos']

In [34]:
def voraz_misioneros(problema):
    """Búsqueda voraz: prioriza solo el menor valor heurístico h(n)."""
    raiz = Nodo(problema.estado_inicial)
    raiz.heuristica = float(problema.heuristica(raiz.estado))

    if problema.es_objetivo(raiz.estado):
        return raiz

    frontera = []
    contador = count()  # desempata nodos con la misma prioridad
    heapq.heappush(frontera, (raiz.heuristica, next(contador), raiz))
    visitados = {raiz.estado}

    while frontera:
        _, _, nodo = heapq.heappop(frontera)

        if problema.es_objetivo(nodo.estado):
            return nodo

        for accion, estado_s in problema.sucesores(nodo.estado):
            if estado_s in visitados:
                continue

            h_s = float(problema.heuristica(estado_s))
            hijo = Nodo(estado_s, padre=nodo, accion=accion, heuristica=h_s)
            visitados.add(estado_s)
            heapq.heappush(frontera, (h_s, next(contador), hijo))

    return None

## Ejemplo: Misioneros y Caníbales con Búsqueda Voraz

Ahora modelamos el problema de **3 misioneros y 3 caníbales** usando la misma idea del ejemplo de ciudades, pero con estados representados por tuplas.

Cada estado será una tripleta `(m_i, c_i, b)` donde:
- `m_i`: número de misioneros en la orilla izquierda.
- `c_i`: número de caníbales en la orilla izquierda.
- `b`: posición del bote (`0` izquierda, `1` derecha).

El estado inicial es `(3,3,0)` y el estado objetivo es `(0,0,1)`.


In [35]:
# definimos una clase del problema con la misma interfaz que necesita voraz_misioneros
class Problema_misioneros:
    def __init__(self, estado_inicial, estados_objetivo):
        self.estado_inicial = estado_inicial
        self.estados_objetivo = set(estados_objetivo)

    def es_objetivo(self, estado):
        return estado in self.estados_objetivo

    def es_valido(self, estado):
        m_i, c_i, b = estado
        m_d = 3 - m_i
        c_d = 3 - c_i

        # cantidades permitidas en cada orilla
        if (m_i < 0) or (m_i > 3) or (c_i < 0) or (c_i > 3):
            return False

        # en la izquierda no puede haber más caníbales que misioneros,
        # salvo que no haya misioneros
        if (m_i > 0) and (c_i > m_i):
            return False

        # la misma restricción debe cumplirse en la derecha
        if (m_d > 0) and (c_d > m_d):
            return False

        return True

    def acciones(self):
        # (misioneros, caníbales) que viajan en el bote
        return [(2,0), (0,2), (1,1), (1,0), (0,1)]

    def resultado(self, estado, accion):
        m_i, c_i, bote = estado
        i, j = accion

        if bote == 0:
            nuevo_estado = (m_i - i, c_i - j, 1)
        else:
            nuevo_estado = (m_i + i, c_i + j, 0)

        if self.es_valido(nuevo_estado):
            return nuevo_estado
        return None

    def sucesores(self, estado):
        sucesores = []
        for accion in self.acciones():
            nuevo_estado = self.resultado(estado, accion)
            if nuevo_estado is not None:
                sucesores.append((accion, nuevo_estado))
        return sucesores

    def costo_accion(self, estado, accion):
        return 1
    

    def heuristica(self, estado):
        # estimamos cuántos cruces útiles faltan según las personas que siguen a la izquierda
        m_i, c_i, bote = estado
        personas_izquierda = m_i + c_i
        if personas_izquierda == 0:
            return 0
        return (personas_izquierda + 1) // 2

    def __str__(self):
        return f'estado inicial={self.estado_inicial}---> estados objetivo:{next(iter(self.estados_objetivo))}'


In [36]:
Pr_m = Problema_misioneros((3,3,0), {(0,0,1)})
str(Pr_m)

'estado inicial=(3, 3, 0)---> estados objetivo:(0, 0, 1)'

In [37]:
Sol_voraz_m = voraz_misioneros(Pr_m)
camino_v_m = [str(i) for i in Sol_voraz_m.reconstruir_camino()]
camino_v_m

['(3, 3, 0)',
 '(3, 1, 1)',
 '(3, 2, 0)',
 '(3, 0, 1)',
 '(3, 1, 0)',
 '(1, 1, 1)',
 '(2, 2, 0)',
 '(0, 2, 1)',
 '(0, 3, 0)',
 '(0, 1, 1)',
 '(1, 1, 0)',
 '(0, 0, 1)']

## Búsqueda A* (A-Star)

La **búsqueda A\*** combina el **costo real acumulado** desde el inicio hasta un nodo $g(n)$ con una **estimación heurística** del costo restante hasta la meta $h(n)$. Su función de evaluación es:
$$
f(n) = g(n) + h(n)
$$
En cada paso, A* expande el nodo de la frontera con **menor $f(n)$**. Así, equilibra “lo que ya costó llegar” ($g$) con “lo que falta según la heurística” ($h$).

---

### Intuición (comparación rápida)
- **UCS** prioriza solo $g(n)$ → encuentra rutas de **menor costo**, pero no “mira” hacia la meta.
- **Voraz** prioriza solo $h(n)$ → **apunta** hacia la meta, pero puede no ser óptimo.
- **A\*** usa $g(n)+h(n)$ → **guía** la búsqueda **y** mantiene **optimalidad** (bajo condiciones sobre $h$).

---

### Requisitos sobre la heurística
- **Admisible**: $h(n) \le h^*(n)$, donde $h^*(n)$ es el costo real óptimo desde $n$ a la meta.  
  - Con **búsqueda en árbol**, A* es **óptimo** si $h$ es admisible.
- **Consistente (monótona)**: $h(n) \le c(n,n') + h(n')$ para toda arista $n \to n'$.  
  - Implica la desigualdad tipo **triángulo**.  
  - Con **búsqueda en grafo** (usando set de cerrados), A* es **óptimo** y **no necesita reabrir** nodos si $h$ es consistente.

> En la práctica: si tu $h$ proviene de una **distancia en línea recta** o de una métrica, casi siempre es consistente.

---

### Propiedades
- **Completa**: Sí, si los costos de arista son $\ge \epsilon > 0$ (no negativos) y hay solución.
- **Óptima**:  
  - **Árbol**: con $h$ admisible.  
  - **Grafo**: con $h$ consistente (o admisible + permitir reabrir nodos cuando se mejora $g$).
- **Complejidad**: en el peor caso, **exponencial** en la profundidad de la solución; en la práctica, muy sensible a la calidad de $h$.
- **Memoria**: mantiene una **frontera** (cola de prioridad por $f$) y un **cerrado/mejor-g**; puede ser intensivo en memoria.

---

### Estructuras típicas (en un diseño claro)
- **Cola de prioridad** (min-heap) ordenada por $f(n)$.
- **`best_g[estado]`**: mejor costo acumulado conocido para cada estado.  
  - Si encuentras un camino con $g$ menor, **actualizas** y vuelves a encolar el nodo.


#### Seudocódigo de Búsqueda A* (A-Star)

```
A*(problema):
    raiz ← Nodo(problema.estado_inicial)
    raiz.costo ← 0
    raiz.heuristica ← h(raiz.estado)
    raiz.f ← raiz.costo + raiz.heuristica
    si problema.es_objetivo(raiz.estado):
        retornar raiz

    frontera ← cola_prioridad_min()         # prioridad = f(n) = g(n)+h(n)
    insertar(frontera, (raiz.f, raiz))
    best_g ← { raiz.estado : 0 }

    mientras frontera no esté vacía:
        (_, n) ← extraer_min(frontera)      # nodo con menor f(n)

        si problema.es_objetivo(n.estado):
            retornar n

        para cada (accion, s) en problema.sucesores(n.estado):
            g_nuevo ← n.costo + costo(n.estado, accion)
            si s no en best_g o g_nuevo < best_g[s]:
                best_g[s] ← g_nuevo
                h_s ← h(s)
                f_s ← g_nuevo + h_s
                hijo ← Nodo(estado=s, padre=n, accion=accion, costo=g_nuevo, heuristica=h_s)
                insertar(frontera, (f_s, hijo))

    retornar fallo   # no existe solución
```


In [38]:
#implementación 
def A_estrella(problema):
    """La función A* que hace la búsqueda teniendo en cuenta el costo g(n) y la heurística h(n)"""
    raiz = Nodo(problema.estado_inicial)
    raiz.costo = 0.0
    raiz.heuristica = float(problema.heuristica(raiz.estado))
    raiz.f = raiz.costo + raiz.heuristica
    if problema.es_objetivo(raiz.estado):
        return raiz
    frontera = []
    contar = count()  # contador para desempatar nodos con la misma f(n)
    heapq.heappush(frontera, (raiz.f, next(contar), raiz))
    best_g = {raiz.estado: 0.0}
    while frontera:
        _, _, nodo = heapq.heappop(frontera)
        if problema.es_objetivo(nodo.estado):
            return nodo
        for accion, s in problema.sucesores(nodo.estado):
            costo_accion = problema.costo_accion(nodo.estado, accion)
            g_s = nodo.costo + costo_accion
            if s not in best_g or g_s < best_g[s]:
                best_g[s] = g_s
                h_s = float(problema.heuristica(s))
                f_s = g_s + h_s
                hijo = Nodo(s, padre=nodo, accion=accion, costo=g_s, heuristica=h_s)
                hijo.f = f_s
                heapq.heappush(frontera, (hijo.f, next(contar), hijo))

## Búsqueda informada para misioneros y caníbales

Ahora resolvemos el mismo problema usando **heurística**, siguiendo la misma idea vista en clase para búsqueda voraz y A*.

### Idea de la heurística

Si en la orilla izquierda quedan `m_i + c_i` personas, y el bote solo puede llevar **2 personas por cruce**, entonces todavía faltan al menos:

$$
h(n) = \left\lceil \frac{m_i + c_i}{2} \right\rceil
$$

cruces útiles para terminar de pasar a todos a la derecha.

- Esta heurística es simple.
- No sobreestima lo que falta, por eso sirve como guía para A*.
- Aunque no cuenta todos los regresos del bote, sí permite priorizar estados que están más cerca de la meta.


In [39]:
def astar_misioneros(problema):
    """Búsqueda A*: combina costo acumulado g(n) y heurística h(n)."""
    raiz = Nodo(
        problema.estado_inicial,
        costo=0.0,
        heuristica=float(problema.heuristica(problema.estado_inicial))
    )

    if problema.es_objetivo(raiz.estado):
        return raiz

    frontera = []
    contador = count()
    heapq.heappush(frontera, (raiz.costo + raiz.heuristica, next(contador), raiz))
    best_g = {raiz.estado: 0.0}

    while frontera:
        _, _, nodo = heapq.heappop(frontera)

        # si ya existe un camino mejor a este estado, descartamos esta copia
        if nodo.costo > best_g.get(nodo.estado, float('inf')):
            continue

        if problema.es_objetivo(nodo.estado):
            return nodo

        for accion, estado_s in problema.sucesores(nodo.estado):
            g_nuevo = nodo.costo + problema.costo_accion(nodo.estado, accion)

            if estado_s not in best_g or g_nuevo < best_g[estado_s]:
                best_g[estado_s] = g_nuevo
                h_s = float(problema.heuristica(estado_s))
                hijo = Nodo(
                    estado_s,
                    padre=nodo,
                    accion=accion,
                    costo=g_nuevo,
                    heuristica=h_s
                )
                f_s = g_nuevo + h_s
                heapq.heappush(frontera, (f_s, next(contador), hijo))

    return None

Sol_astar_m = astar_misioneros(Pr_m)
print('Camino A*:')
print(Sol_astar_m.reconstruir_camino())
print('Acciones A*:')
print(Sol_astar_m.reconstruir_acciones())


Camino A*:
[(3, 3, 0), (3, 1, 1), (3, 2, 0), (3, 0, 1), (3, 1, 0), (1, 1, 1), (2, 2, 0), (0, 2, 1), (0, 3, 0), (0, 1, 1), (1, 1, 0), (0, 0, 1)]
Acciones A*:
[(0, 2), (0, 1), (0, 2), (0, 1), (2, 0), (1, 1), (2, 0), (0, 1), (0, 2), (1, 0), (1, 1)]


### ¿Cómo se construye la solución con A*?

A* evalúa cada estado con:

$$
f(n) = g(n) + h(n)
$$

donde:

- `g(n)` es el número de cruces hechos hasta llegar al estado.
- `h(n)` estima cuántos cruces útiles faltan.
- `f(n)` combina ambas ideas para decidir qué estado explorar primero.

En este problema, eso significa que A* no solo mira cuál estado parece más cercano a la meta, sino también cuánto ha costado llegar hasta allí. Por eso suele encontrar una solución mejor guiada que la búsqueda voraz.


In [40]:
camino_astar = Sol_astar_m.reconstruir_camino()
acciones_astar = Sol_astar_m.reconstruir_acciones()

print('Solución final con A*:')
for i, estado in enumerate(camino_astar):
    print(estado)
    if i < len(acciones_astar):
        print('  --', acciones_astar[i], '-->')


Solución final con A*:
(3, 3, 0)
  -- (0, 2) -->
(3, 1, 1)
  -- (0, 1) -->
(3, 2, 0)
  -- (0, 2) -->
(3, 0, 1)
  -- (0, 1) -->
(3, 1, 0)
  -- (2, 0) -->
(1, 1, 1)
  -- (1, 1) -->
(2, 2, 0)
  -- (2, 0) -->
(0, 2, 1)
  -- (0, 1) -->
(0, 3, 0)
  -- (0, 2) -->
(0, 1, 1)
  -- (1, 0) -->
(1, 1, 0)
  -- (1, 1) -->
(0, 0, 1)


## Ejercicio: El 8-Puzzle

El **8-Puzzle** es un clásico problema de búsqueda: el objetivo es transformar una configuración inicial de fichas en una configuración objetivo, moviendo la celda vacía en un tablero de 3×3. Cada movimiento desplaza la celda vacía (en azul) hacia arriba, abajo, izquierda o derecha, intercambiándola con una ficha adyacente.


<p align='center'>

<img src="PUZZLE8.PNG" alt="8-Puzzle" width="450"/>

</p>

### Tarea: 8-Puzzle – Algoritmos de Búsqueda

El objetivo de esta actividad es integrar **modelado**, **implementación** y **análisis** de distintos algoritmos de búsqueda sobre el clásico problema del **8-Puzzle**. La tarea está organizada en varias etapas para promover tanto la parte técnica como la comprensión teórica.

---

### 1. Modelado del problema
- Representa el espacio de estados del 8-Puzzle:
  - Cada estado puede ser una **tupla de 9 enteros** (ej. `(1,2,3,4,5,6,7,8,0)`, donde `0` es el espacio en blanco).
  - Las acciones son los movimientos del espacio en blanco: **Arriba**, **Abajo**, **Izquierda**, **Derecha**.
- Genera **sucesores dinámicamente** según la posición del espacio en blanco.
- Define el **costo** de cada acción (por defecto, costo = 1).

---

### 2. Implementación de algoritmos
Implementa los siguientes algoritmos sobre el 8-Puzzle:

- **Búsqueda no informada:**
  - BFS (Breadth-First Search)
  - DFS (Depth-First Search)
  - UCS (Uniform-Cost Search)

- **Búsqueda informada:**
  - Voraz (Greedy Best-First Search)
  - A* (A-Star Search)

**Requisitos:**
- Usa la **misma estructura de datos** para todos los algoritmos (`Nodo`, `Problema`, etc.).
- Implementa al menos **dos heurísticas**:
  1. Número de fichas mal colocadas.
  2. Suma de distancias Manhattan.

---

### 3. Experimentos
- Selecciona **3 configuraciones iniciales** con distinta dificultad (fácil, media, difícil).
- Para cada algoritmo y configuración, mide:
  - Número de nodos expandidos.
  - Longitud del camino solución.
  - Tiempo de ejecución aproximado.

Presenta estos datos en una **tabla comparativa** por instancia y algoritmo.

---

### 4. Análisis y discusión
- Compara las **heurísticas**:  
  ¿Cómo afectan el rendimiento de la búsqueda Voraz y A*?
- Compara los **algoritmos**:
  - ¿Cuál encuentra siempre la solución óptima?
  - ¿Cuál explora menos nodos?
  - ¿Cuándo la búsqueda Voraz deja de ser óptima?
- Reflexiona sobre la relación entre **costo acumulado** y **profundidad** de la solución.




## Solución guiada: 8-Puzzle

Vamos a resolver el ejercicio siguiendo la misma idea de clase: primero **modelamos el problema**, luego definimos **costos y heurísticas**, y finalmente aplicamos los algoritmos de búsqueda para comparar su comportamiento.

### Representación del estado

Representaremos cada tablero como una **tupla de 9 números**.

Por ejemplo:

`(1,2,3,4,5,6,7,8,0)`

significa que las fichas están ordenadas y el `0` representa el espacio vacío.

La meta será precisamente ese estado ordenado.


In [41]:
from collections import deque
import time

def imprimir_tablero(estado):
    """Muestra un estado del 8-puzzle con forma de tablero 3x3."""
    for i in range(0, 9, 3):
        fila = estado[i:i+3]
        print(' '.join(str(x) if x != 0 else '_' for x in fila))
    print()

class Cola:
    def __init__(self):
        self.items = deque()

    def encolar(self, x):
        self.items.append(x)

    def desencolar(self):
        return self.items.popleft()

    def vacia(self):
        return len(self.items) == 0

class Pila:
    def __init__(self):
        self.items = []

    def apilar(self, x):
        self.items.append(x)

    def desapilar(self):
        return self.items.pop()

    def vacia(self):
        return len(self.items) == 0


In [42]:
class Problema8Puzzle:
    """Modela el 8-puzzle usando la misma idea de la clase Problema."""
    def __init__(self, estado_inicial, estado_objetivo=(1,2,3,4,5,6,7,8,0), tipo_heuristica='manhattan'):
        self.estado_inicial = estado_inicial
        self.estado_objetivo = estado_objetivo
        self.tipo_heuristica = tipo_heuristica
        self.posiciones_objetivo = {valor: i for i, valor in enumerate(estado_objetivo)}

    def _estado(self, x):
        return x.estado if hasattr(x, 'estado') else x

    def es_objetivo(self, x):
        return self._estado(x) == self.estado_objetivo

    def acciones(self, estado):
        posicion_vacia = estado.index(0)
        fila, columna = divmod(posicion_vacia, 3)
        acciones = []

        if fila > 0:
            acciones.append('Arriba')
        if fila < 2:
            acciones.append('Abajo')
        if columna > 0:
            acciones.append('Izquierda')
        if columna < 2:
            acciones.append('Derecha')

        return acciones

    def resultado(self, estado, accion):
        posicion_vacia = estado.index(0)

        if accion == 'Arriba':
            nueva_posicion = posicion_vacia - 3
        elif accion == 'Abajo':
            nueva_posicion = posicion_vacia + 3
        elif accion == 'Izquierda':
            nueva_posicion = posicion_vacia - 1
        else:
            nueva_posicion = posicion_vacia + 1

        nuevo_estado = list(estado)
        nuevo_estado[posicion_vacia], nuevo_estado[nueva_posicion] = nuevo_estado[nueva_posicion], nuevo_estado[posicion_vacia]
        return tuple(nuevo_estado)

    def sucesores(self, estado):
        sucesores = []
        for accion in self.acciones(estado):
            sucesores.append((accion, self.resultado(estado, accion)))
        return sucesores

    def costo_accion(self, estado, accion):
        return 1

    def heuristica(self, estado):
        if self.tipo_heuristica == 'mal_colocadas':
            return sum(1 for i, valor in enumerate(estado) if valor != 0 and valor != self.estado_objetivo[i])

        distancia = 0
        for i, valor in enumerate(estado):
            if valor == 0:
                continue

            pos_obj = self.posiciones_objetivo[valor]
            fila_actual, col_actual = divmod(i, 3)
            fila_obj, col_obj = divmod(pos_obj, 3)
            distancia += abs(fila_actual - fila_obj) + abs(col_actual - col_obj)

        return distancia

    def __str__(self):
        return f'estado inicial={self.estado_inicial} ---> estado objetivo={self.estado_objetivo}'


### Heurísticas que vamos a usar

Siguiendo la guía de la actividad, vamos a trabajar con dos heurísticas clásicas:

1. **Fichas mal colocadas**: cuenta cuántas fichas no están en su posición final.
2. **Distancia Manhattan**: suma cuántos pasos horizontales y verticales le faltan a cada ficha para llegar a su lugar.

La heurística de Manhattan suele ser mejor porque usa más información del tablero.


In [43]:
estado_ejemplo = (1,2,3,4,0,6,7,5,8)
Pr_ejemplo_h1 = Problema8Puzzle(estado_ejemplo, tipo_heuristica='mal_colocadas')
Pr_ejemplo_h2 = Problema8Puzzle(estado_ejemplo, tipo_heuristica='manhattan')

print('Tablero de ejemplo:')
imprimir_tablero(estado_ejemplo)
print('Heurística mal colocadas =', Pr_ejemplo_h1.heuristica(estado_ejemplo))
print('Heurística Manhattan =', Pr_ejemplo_h2.heuristica(estado_ejemplo))


Tablero de ejemplo:
1 2 3
4 _ 6
7 5 8

Heurística mal colocadas = 2
Heurística Manhattan = 2


In [44]:
# implementamos los algoritmos no informados con el mismo estilo visto en clase
def BFS(problema):
    """Búsqueda en anchura: explora por niveles."""
    c_n = 0
    raiz = Nodo(problema.estado_inicial)

    if problema.es_objetivo(raiz):
        return (raiz, 0)

    frontera = Cola()
    frontera.encolar(raiz)
    visitados = {raiz.estado}

    while not frontera.vacia():
        c_n += 1
        nodo = frontera.desencolar()

        for accion, estado_s in problema.sucesores(nodo.estado):
            if estado_s not in visitados:
                hijo = Nodo(estado_s, padre=nodo, accion=accion)
                if problema.es_objetivo(hijo):
                    return (hijo, c_n)
                frontera.encolar(hijo)
                visitados.add(hijo.estado)

    return (None, c_n)

def DFS(problema):
    """Búsqueda en profundidad: avanza por un camino antes de retroceder."""
    c_n = 0
    raiz = Nodo(problema.estado_inicial)

    if problema.es_objetivo(raiz):
        return (raiz, 0)

    frontera = Pila()
    frontera.apilar(raiz)
    visitados = {raiz.estado}

    while not frontera.vacia():
        c_n += 1
        nodo = frontera.desapilar()

        for accion, estado_s in problema.sucesores(nodo.estado):
            if estado_s not in visitados:
                hijo = Nodo(estado_s, padre=nodo, accion=accion)
                if problema.es_objetivo(hijo):
                    return (hijo, c_n)
                frontera.apilar(hijo)
                visitados.add(hijo.estado)

    return (None, c_n)

def UCS(problema):
    """Búsqueda de costo uniforme: prioriza el menor costo acumulado g(n)."""
    raiz = Nodo(problema.estado_inicial)

    if problema.es_objetivo(raiz):
        return (raiz, 0)

    frontera = []
    contar = count()
    heapq.heappush(frontera, (0.0, next(contar), raiz))
    best_g = {raiz.estado: 0.0}
    c_n = 0

    while frontera:
        g, _, nodo = heapq.heappop(frontera)

        if g > best_g.get(nodo.estado, float('inf')):
            continue

        c_n += 1

        if problema.es_objetivo(nodo):
            nodo.costo = g
            return (nodo, c_n)

        for accion, estado_s in problema.sucesores(nodo.estado):
            g_nuevo = g + problema.costo_accion(nodo.estado, accion)

            if (estado_s not in best_g) or (g_nuevo < best_g[estado_s]):
                best_g[estado_s] = g_nuevo
                hijo = Nodo(estado_s, padre=nodo, accion=accion, costo=g_nuevo)
                heapq.heappush(frontera, (g_nuevo, next(contar), hijo))

    return (None, c_n)


In [45]:
# implementamos los algoritmos informados con la misma estructura usada arriba
def voraz_puzzle(problema):
    """Búsqueda voraz: usa solo h(n)."""
    raiz = Nodo(problema.estado_inicial)
    raiz.heuristica = float(problema.heuristica(raiz.estado))

    if problema.es_objetivo(raiz.estado):
        return (raiz, 0)

    frontera = []
    contador = count()
    heapq.heappush(frontera, (raiz.heuristica, next(contador), raiz))
    visitados = {raiz.estado}
    c_n = 0

    while frontera:
        _, _, nodo = heapq.heappop(frontera)
        c_n += 1

        if problema.es_objetivo(nodo.estado):
            return (nodo, c_n)

        for accion, estado_s in problema.sucesores(nodo.estado):
            if estado_s in visitados:
                continue

            h_s = float(problema.heuristica(estado_s))
            hijo = Nodo(estado_s, padre=nodo, accion=accion, heuristica=h_s)
            visitados.add(estado_s)
            heapq.heappush(frontera, (h_s, next(contador), hijo))

    return (None, c_n)

def astar_puzzle(problema):
    """Búsqueda A*: combina g(n) + h(n)."""
    raiz = Nodo(
        problema.estado_inicial,
        costo=0.0,
        heuristica=float(problema.heuristica(problema.estado_inicial))
    )

    if problema.es_objetivo(raiz.estado):
        return (raiz, 0)

    frontera = []
    contador = count()
    heapq.heappush(frontera, (raiz.costo + raiz.heuristica, next(contador), raiz))
    best_g = {raiz.estado: 0.0}
    c_n = 0

    while frontera:
        _, _, nodo = heapq.heappop(frontera)

        if nodo.costo > best_g.get(nodo.estado, float('inf')):
            continue

        c_n += 1

        if problema.es_objetivo(nodo.estado):
            return (nodo, c_n)

        for accion, estado_s in problema.sucesores(nodo.estado):
            g_nuevo = nodo.costo + problema.costo_accion(nodo.estado, accion)

            if estado_s not in best_g or g_nuevo < best_g[estado_s]:
                best_g[estado_s] = g_nuevo
                h_s = float(problema.heuristica(estado_s))
                hijo = Nodo(
                    estado_s,
                    padre=nodo,
                    accion=accion,
                    costo=g_nuevo,
                    heuristica=h_s
                )
                f_s = g_nuevo + h_s
                heapq.heappush(frontera, (f_s, next(contador), hijo))

    return (None, c_n)


### Instancias de prueba

Para no alejarnos de lo que vimos en clase, vamos a usar tres instancias **solubles** y de dificultad creciente:

- **Fácil**: está a un solo movimiento de la meta.
- **Media**: requiere pocos movimientos, pero ya obliga a explorar.
- **Difícil**: permite ver mejor la diferencia entre búsqueda no informada y búsqueda informada.


In [46]:
instancias = {
    'facil': (1,2,3,4,5,6,7,0,8),
    'media': (1,2,3,4,0,6,7,5,8),
    'dificil': (1,3,6,5,0,2,4,7,8)
}

print('Instancia difícil:')
imprimir_tablero(instancias['dificil'])

Pr_dificil = Problema8Puzzle(instancias['dificil'], tipo_heuristica='manhattan')
Sol_astar_dificil, nodos_astar_dificil = astar_puzzle(Pr_dificil)

print('Número de nodos expandidos por A*:', nodos_astar_dificil)
print('Acciones encontradas por A*:', Sol_astar_dificil.reconstruir_acciones())


Instancia difícil:
1 3 6
5 _ 2
4 7 8

Número de nodos expandidos por A*: 13
Acciones encontradas por A*: ['Derecha', 'Arriba', 'Izquierda', 'Abajo', 'Izquierda', 'Abajo', 'Derecha', 'Derecha']


In [47]:
camino_astar_8 = Sol_astar_dificil.reconstruir_camino()
acciones_astar_8 = Sol_astar_dificil.reconstruir_acciones()

print('Solución paso a paso con A* y heurística Manhattan:')
for i, estado in enumerate(camino_astar_8):
    print(f'Paso {i}:')
    imprimir_tablero(estado)
    if i < len(acciones_astar_8):
        print('Acción:', acciones_astar_8[i])
        print()


Solución paso a paso con A* y heurística Manhattan:
Paso 0:
1 3 6
5 _ 2
4 7 8

Acción: Derecha

Paso 1:
1 3 6
5 2 _
4 7 8

Acción: Arriba

Paso 2:
1 3 _
5 2 6
4 7 8

Acción: Izquierda

Paso 3:
1 _ 3
5 2 6
4 7 8

Acción: Abajo

Paso 4:
1 2 3
5 _ 6
4 7 8

Acción: Izquierda

Paso 5:
1 2 3
_ 5 6
4 7 8

Acción: Abajo

Paso 6:
1 2 3
4 5 6
_ 7 8

Acción: Derecha

Paso 7:
1 2 3
4 5 6
7 _ 8

Acción: Derecha

Paso 8:
1 2 3
4 5 6
7 8 _



In [48]:
def ejecutar_experimento(nombre_algoritmo, algoritmo, estado_inicial, heuristica=None):
    if heuristica is None:
        problema = Problema8Puzzle(estado_inicial)
    else:
        problema = Problema8Puzzle(estado_inicial, tipo_heuristica=heuristica)

    t0 = time.perf_counter()
    solucion, nodos = algoritmo(problema)
    t1 = time.perf_counter()

    if solucion is None:
        longitud = None
    else:
        longitud = len(solucion.reconstruir_acciones())

    return {
        'algoritmo': nombre_algoritmo,
        'heuristica': heuristica if heuristica is not None else '-',
        'nodos': nodos,
        'longitud': longitud,
        'tiempo': round(t1 - t0, 5)
    }

def imprimir_tabla(resultados):
    encabezado = f"{'Algoritmo':<12}{'Heurística':<18}{'Nodos':<12}{'Longitud':<12}{'Tiempo(s)':<10}"
    print(encabezado)
    print('-' * len(encabezado))

    for r in resultados:
        print(f"{r['algoritmo']:<12}{r['heuristica']:<18}{r['nodos']:<12}{str(r['longitud']):<12}{r['tiempo']:<10}")

algoritmos = [
    ('BFS', BFS, None),
    ('DFS', DFS, None),
    ('UCS', UCS, None),
    ('Voraz', voraz_puzzle, 'mal_colocadas'),
    ('Voraz', voraz_puzzle, 'manhattan'),
    ('A*', astar_puzzle, 'mal_colocadas'),
    ('A*', astar_puzzle, 'manhattan')
]

for nombre_instancia, estado in instancias.items():
    print(f'\nResultados para la instancia: {nombre_instancia}')
    resultados = []

    for nombre_algoritmo, algoritmo, heuristica in algoritmos:
        resultados.append(ejecutar_experimento(nombre_algoritmo, algoritmo, estado, heuristica))

    imprimir_tabla(resultados)



Resultados para la instancia: facil
Algoritmo   Heurística        Nodos       Longitud    Tiempo(s) 
----------------------------------------------------------------
BFS         -                 1           1           1e-05     
DFS         -                 1           1           1e-05     
UCS         -                 4           1           1e-05     
Voraz       mal_colocadas     2           1           1e-05     
Voraz       manhattan         2           1           1e-05     
A*          mal_colocadas     2           1           1e-05     
A*          manhattan         2           1           1e-05     

Resultados para la instancia: media
Algoritmo   Heurística        Nodos       Longitud    Tiempo(s) 
----------------------------------------------------------------
BFS         -                 3           2           1e-05     
DFS         -                 135585      49284       0.20082   
UCS         -                 9           2           3e-05     
Voraz       mal_

### Lectura de los resultados

Al ejecutar la tabla, normalmente se observa algo como esto:

- **BFS** encuentra soluciones óptimas en número de pasos cuando todos los costos son 1, pero puede expandir muchos nodos.
- **DFS** puede encontrar una solución, pero no garantiza que sea la mejor. De hecho, a veces recorre caminos muy largos antes de llegar a la meta.
- **UCS** también encuentra solución óptima porque aquí todos los movimientos cuestan 1, aunque puede explorar más que A*.
- **Voraz** suele expandir menos nodos, pero no siempre devuelve la ruta más corta.
- **A*** combina el costo acumulado con la heurística, por eso normalmente mantiene la solución óptima y además reduce la exploración.
- Entre las dos heurísticas, **Manhattan** suele funcionar mejor que **mal colocadas**, porque distingue con más detalle qué tan lejos está cada ficha de su posición final.

### Idea para explicar esto en clase

Puedes decirlo así: en el 8-puzzle, `BFS` y `UCS` garantizan solución óptima, pero `A*` logra esa misma meta explorando menos porque usa información adicional del problema. `DFS` no es conveniente cuando queremos caminos cortos, y la búsqueda voraz puede ser rápida, pero sacrifica optimalidad.
